In [2]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/processed/encounters_clean_step2.csv", low_memory=False)
df.shape

(99340, 49)

In [4]:
admission_type_map = {1: "Emergency", 2: "Urgent", 3: "Elective", 4: "Newborn", 7: "Trauma Center"}
df["admission_type"] = df["admission_type_id"].map(admission_type_map).fillna("Unknown")
df["admission_type"].value_counts()

admission_type
Emergency        52369
Elective         18667
Urgent           18132
Unknown          10144
Trauma Center       18
Newborn             10
Name: count, dtype: int64

In [5]:
discharge_map = {
    1: "Home",
    6: "Home with health service", 8: "Home with health service",
    3: "Skilled nursing facility", 4: "Skilled nursing facility", 24: "Skilled nursing facility",
    15: "Rehab / long-term care", 22: "Rehab / long-term care", 23: "Rehab / long-term care",
    2: "Other hospital / facility", 5: "Other hospital / facility", 9: "Other hospital / facility",
    10: "Other hospital / facility", 12: "Other hospital / facility", 16: "Other hospital / facility",
    17: "Other hospital / facility", 27: "Other hospital / facility", 28: "Other hospital / facility",
    7: "Left against medical advice",
}
df["discharge_group"] = df["discharge_disposition_id"].map(discharge_map).fillna("Unknown")

df.groupby("discharge_group")["readmitted_30d"].agg(["count", "mean"]).sort_values("mean", ascending=False).round(3)

,count,mean
discharge_group,,
Rehab / long-term care,2467,0.247
Other hospital / facility,3511,0.185
Skilled nursing facility,14817,0.146
Left against medical advice,623,0.144
Home with health service,13010,0.127
Unknown,4680,0.118
Home,60232,0.093


In [6]:
source_map = {7: "Emergency room", 1: "Referral", 2: "Referral", 3: "Referral",
              4: "Transfer", 5: "Transfer", 6: "Transfer", 10: "Transfer", 18: "Transfer",
              22: "Transfer", 25: "Transfer", 26: "Transfer"}
df["admission_source"] = df["admission_source_id"].map(source_map).fillna("Other / unknown")
df["admission_source"].value_counts()

admission_source
Emergency room     55848
Referral           30433
Other / unknown     6874
Transfer            6185
Name: count, dtype: int64

In [7]:
print("Number of specialties:", df["medical_specialty"].nunique())

top10 = df["medical_specialty"].value_counts().head(10).index
df["specialty_group"] = df["medical_specialty"].where(df["medical_specialty"].isin(top10), "Other")
df["specialty_group"].value_counts()

Number of specialties: 73


specialty_group
Unknown                       48614
InternalMedicine              14237
Other                          8199
Emergency/Trauma               7419
Family/GeneralPractice         7252
Cardiology                     5278
Surgery-General                3059
Nephrology                     1539
Orthopedics                    1392
Orthopedics-Reconstructive     1230
Radiologist                    1121
Name: count, dtype: int64

In [8]:
print("Unique primary diagnosis codes:", df["diag_1"].nunique())

def icd9_group(code):
    if pd.isna(code):
        return "Missing"
    code = str(code)
    if code.startswith(("V", "E")):          # V/E codes = check-ups, external causes
        return "Other"
    value = float(code)
    if 250 <= value < 251:
        return "Diabetes"
    if 390 <= value <= 459 or int(value) == 785:
        return "Circulatory"
    if 460 <= value <= 519 or int(value) == 786:
        return "Respiratory"
    if 520 <= value <= 579 or int(value) == 787:
        return "Digestive"
    if 800 <= value <= 999:
        return "Injury"
    if 710 <= value <= 739:
        return "Musculoskeletal"
    if 580 <= value <= 629 or int(value) == 788:
        return "Genitourinary"
    if 140 <= value <= 239:
        return "Neoplasms"
    return "Other"

for col in ["diag_1", "diag_2", "diag_3"]:
    df[col + "_group"] = df[col].apply(icd9_group)

df = df.rename(columns={"diag_1_group": "primary_diagnosis"})
df["primary_diagnosis"].value_counts()

Unique primary diagnosis codes: 715


primary_diagnosis
Circulatory        29680
Other              17793
Respiratory        13934
Digestive           9333
Diabetes            8661
Injury              6851
Genitourinary       5002
Musculoskeletal     4935
Neoplasms           3131
Missing               20
Name: count, dtype: int64

In [9]:
df["total_prior_visits"] = df["number_outpatient"] + df["number_emergency"] + df["number_inpatient"]

df["prior_inpatient_band"] = pd.cut(df["number_inpatient"],
                                    bins=[-1, 0, 1, 2, 100],
                                    labels=["0", "1", "2", "3+"])

df.groupby("prior_inpatient_band", observed=True)["readmitted_30d"].agg(["count", "mean"]).round(3)

,count,mean
prior_inpatient_band,,
0,66242,0.086
1,18984,0.133
2,7300,0.179
3+,6814,0.264


In [10]:
df["los_band"] = pd.cut(df["time_in_hospital"], bins=[0, 2, 4, 7, 14],
                        labels=["1-2 days", "3-4 days", "5-7 days", "8-14 days"])

df.groupby("los_band", observed=True)["readmitted_30d"].agg(["count", "mean"]).round(3)

,count,mean
los_band,,
1-2 days,30713,0.093
3-4 days,31116,0.113
5-7 days,22800,0.127
8-14 days,14711,0.138


In [11]:
df["a1c_tested"] = (df["A1Cresult"] != "Not tested").astype(int)
df["a1c_high"] = df["A1Cresult"].isin([">7", ">8"]).astype(int)
df["med_changed"] = (df["change"] == "Ch").astype(int)

df["hba1c_management"] = np.select(
    [df["a1c_tested"] == 0, df["a1c_high"] == 0, df["med_changed"] == 1],
    ["1. Not tested", "2. Tested - normal", "3. High - medication changed"],
    default="4. High - no medication change")

df.groupby("hba1c_management")["readmitted_30d"].agg(["count", "mean"]).round(3)

,count,mean
hba1c_management,,
1. Not tested,82506,0.117
2. Tested - normal,4922,0.098
3. High - medication changed,7227,0.103
4. High - no medication change,4685,0.095


In [12]:
med_cols = ["metformin", "repaglinide", "nateglinide", "chlorpropamide", "glimepiride",
            "acetohexamide", "glipizide", "glyburide", "tolbutamide", "pioglitazone",
            "rosiglitazone", "acarbose", "miglitol", "troglitazone", "tolazamide", "insulin",
            "glyburide-metformin", "glipizide-metformin", "glimepiride-pioglitazone",
            "metformin-rosiglitazone", "metformin-pioglitazone"]

df["n_diabetes_meds"] = (df[med_cols] != "No").sum(axis=1)
df["n_med_changes"] = df[med_cols].isin(["Up", "Down"]).sum(axis=1)
df["on_insulin"] = (df["insulin"] != "No").astype(int)
df["on_diabetes_med"] = (df["diabetesMed"] == "Yes").astype(int)

df[["n_diabetes_meds", "n_med_changes", "on_insulin", "on_diabetes_med"]].describe().round(2)

,n_diabetes_meds,n_med_changes,on_insulin,on_diabetes_med
count,99340.00,99340.00,99340.00,99340.00
mean,1.19,0.29,0.53,0.77
std,0.92,0.49,0.50,0.42
min,0.00,0.00,0.00,0.00
25%,1.00,0.00,0.00,1.00
50%,1.00,0.00,1.00,1.00
75%,2.00,1.00,1.00,1.00
max,6.00,4.00,1.00,1.00


In [13]:
df = df.sort_values(["patient_nbr", "encounter_id"])
df["encounter_seq"] = df.groupby("patient_nbr").cumcount() + 1
df["is_first_encounter"] = (df["encounter_seq"] == 1).astype(int)

print(df[["patient_nbr", "encounter_id", "encounter_seq", "is_first_encounter"]].head(6))
print("First encounters (= unique patients):", df["is_first_encounter"].sum())
print(df.groupby(df["encounter_seq"].clip(upper=4))["readmitted_30d"].mean().round(3))

       patient_nbr  encounter_id  encounter_seq  is_first_encounter
4199           135      24437208              1                   1
4695           135      26264286              2                   0
5710           378      29758806              1                   1
66045          729     189899286              1                   1
17169          774      64331490              1                   1
2238           927      14824206              1                   1
First encounters (= unique patients): 69987
encounter_seq
1    0.090
2    0.142
3    0.174
4    0.239
Name: readmitted_30d, dtype: float64


In [14]:
keep = [
    "encounter_id", "patient_nbr", "encounter_seq", "is_first_encounter",
    "race", "gender", "age", "age_years",
    "admission_type", "admission_source", "discharge_group", "specialty_group", "payer_code",
    "time_in_hospital", "los_band", "num_lab_procedures", "num_procedures", "num_medications",
    "number_outpatient", "number_emergency", "number_inpatient", "total_prior_visits",
    "prior_inpatient_band", "number_diagnoses",
    "primary_diagnosis", "diag_2_group", "diag_3_group",
    "A1Cresult", "max_glu_serum", "a1c_tested", "a1c_high", "hba1c_management",
    "n_diabetes_meds", "n_med_changes", "on_insulin", "med_changed", "on_diabetes_med",
    "readmitted", "readmitted_30d",
]
final = df[keep]
print(final.shape, "| missing values:", final.isna().sum().sum())

final.to_csv("../data/processed/encounters_features.csv", index=False)

(99340, 39) | missing values: 0
